# SMS Spam Detection Using Support Vector Machine (SVM)

**Goal:** classify SMS messages as **spam** or **ham (not spam)** using TF-IDF features and a Linear SVM.

**Dataset:** [SMS Spam Collection (Kaggle)](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

> In Google Colab, upload the dataset CSV when prompted. The code also works locally if `spam.csv` or `SMS.csv` is in the notebook's working directory.

In [ ]:
# Install only if needed (Google Colab/Jupyter)
# %pip -q install pandas numpy matplotlib seaborn scikit-learn wordcloud

## 1. Import libraries

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Load the dataset
Run this cell and select the CSV file if it is not already in the working directory.

In [ ]:
# Google Colab upload (skipped automatically outside Colab)
candidate_files = ["spam.csv", "SMS.csv", "SMSSpamCollection"]

dataset_path = next((name for name in candidate_files if os.path.exists(name)), None)

if dataset_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload a CSV dataset file.")
        dataset_path = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. Place spam.csv or SMS.csv beside this notebook."
        )

# Kaggle's common dataset has columns v1 (label), v2 (message), plus unused columns.
raw_df = pd.read_csv(dataset_path, encoding="latin-1")
if {"v1", "v2"}.issubset(raw_df.columns):
    df = raw_df[["v1", "v2"]].copy()
    df.columns = ["label", "message"]
elif {"label", "message"}.issubset(raw_df.columns):
    df = raw_df[["label", "message"]].copy()
elif raw_df.shape[1] >= 2:
    df = raw_df.iloc[:, :2].copy()
    df.columns = ["label", "message"]
else:
    raise ValueError("CSV must contain a label column and a message column.")

df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str)
df = df[df["label"].isin(["ham", "spam"])].dropna(subset=["message"])
df = df.drop_duplicates(subset=["label", "message"]).reset_index(drop=True)

if df.empty:
    raise ValueError("No valid ham/spam rows found. Check the dataset format.")

print(f"Loaded file: {dataset_path}")
print(f"Dataset shape: {df.shape}")
display(df.head())
print("\\nLabel counts:")
print(df["label"].value_counts())

## 3. Explore the dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="label", order=["ham", "spam"], ax=axes[0])
axes[0].set_title("Number of Ham and Spam Messages")
axes[0].set_xlabel("Message type")
axes[0].set_ylabel("Count")

df["character_count"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()
sns.histplot(data=df, x="word_count", hue="label", bins=30, element="step",
             stat="count", common_norm=False, ax=axes[1])
axes[1].set_title("Message Length by Word Count")
axes[1].set_xlabel("Number of words")

plt.tight_layout()
plt.show()

print("Missing values:\\n", df[["label", "message"]].isna().sum())
print("Duplicate rows remaining:", df.duplicated(subset=["label", "message"]).sum())

## 4. Prepare text and split the data

The train/test split is performed **before fitting the vectorizer**, and TF-IDF is kept inside the pipeline to avoid leaking information from the test set into training.

In [ ]:
# Keep the original message text; TF-IDF will handle tokenization and feature extraction.
X = df["message"]
y = df["label"].map({"ham": 0, "spam": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training label distribution:")
print(y_train.value_counts(normalize=True).rename("proportion"))

## 5. Train the SVM model

In [ ]:
model_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        max_features=10000,
        sublinear_tf=True
    )),
    ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
])

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"], zero_division=0))

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Ham", "Spam"])
disp.plot(cmap="Blues", values_format="d")
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.show()

print("Rows = actual labels; columns = predicted labels.")

## 7. Hyperparameter tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [5000, 10000],
    "svm__C": [0.1, 1, 10]
}

grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validation F1-score: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
tuned_pred = best_model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, tuned_pred):.4f}")
print(f"Test precision: {precision_score(y_test, tuned_pred, zero_division=0):.4f}")
print(f"Test recall: {recall_score(y_test, tuned_pred, zero_division=0):.4f}")
print(f"Test F1-score: {f1_score(y_test, tuned_pred, zero_division=0):.4f}")
print(classification_report(y_test, tuned_pred, target_names=["Ham", "Spam"], zero_division=0))

## 8. Cross-validation

In [ ]:
cv_scores = cross_val_score(
    best_model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
)
print("F1 scores for each fold:", np.round(cv_scores, 4))
print(f"Mean CV F1-score: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")

## 9. Predict a new SMS message

In [ ]:
# Use the tuned model for new predictions.
def predict_sms(message):
    """Predict whether a single SMS message is Spam or Ham."""
    if not isinstance(message, str) or not message.strip():
        raise ValueError("Please enter a non-empty SMS message.")

    prediction = best_model.predict([message])[0]
    label = "Spam" if prediction == 1 else "Ham (Not Spam)"
    return label

examples = [
    "Congratulations! You have won a free iPhone. Click here now!",
    "Hi, are we still meeting tomorrow at 3 PM?"
]

for sms in examples:
    print(f"Message: {sms}")
    print(f"Prediction: {predict_sms(sms)}\\n")

## 10. Conclusion

This notebook builds an SMS spam classifier using TF-IDF text features and a Linear Support Vector Machine. It evaluates the model with accuracy, precision, recall, F1-score, a confusion matrix, and cross-validation, then tunes model settings using the training data only.

**Note:** Run the cells from top to bottom. Report the metrics produced by your own run; results can vary with dataset versions and settings.